# Text transformer 

Develop a simple transformer model for processing text data. The objective is to build on this transformer a binary classifier that can determine whether a given text belongs to a certain category (e.g., spam vs. not spam, positive vs. negative sentiment).

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets.mnist import MNIST
from torchvision.transforms import ToTensor
from tqdm import tqdm, trange

In [2]:
class SimpleTokenizer:
    def __init__(self, texts):
        vocab = set()
        for t in texts:
            vocab.update(t.lower().split())
        self.word2idx = {w: i+2 for i, w in enumerate(vocab)}
        self.word2idx["<PAD>"] = 0
        self.word2idx["<CLS>"] = 1
        self.idx2word = {i: w for w, i in self.word2idx.items()}

    def encode(self, text, max_len):
        tokens = ["<CLS>"] + text.lower().split()
        ids = [self.word2idx.get(t, 0) for t in tokens]
        ids = ids[:max_len]
        ids += [0] * (max_len - len(ids))
        return torch.tensor(ids)

In [3]:
def get_positional_embeddings(seq_len, d):
    pos = torch.zeros(seq_len, d)
    for i in range(seq_len):
        for j in range(d):
            pos[i, j] = (
                np.sin(i / (10000 ** (j / d)))
                if j % 2 == 0
                else np.cos(i / (10000 ** ((j - 1) / d)))
            )
    return pos

In [ ]:
# Multi-Head Self-Attention
class MyMSA(nn.Module):
    def __init__(self, d, n_heads=2):
        super(MyMSA, self).__init__()
        self.d = d
        self.n_heads = n_heads

        assert d % n_heads == 0, f"Can't divide dimension {d} into {n_heads} heads"

        d_head = int(d / n_heads)
        self.q_mappings = nn.ModuleList(
            [nn.Linear(d_head, d_head) for _ in range(self.n_heads)]
        )
        self.k_mappings = nn.ModuleList(
            [nn.Linear(d_head, d_head) for _ in range(self.n_heads)]
        )
        self.v_mappings = nn.ModuleList(
            [nn.Linear(d_head, d_head) for _ in range(self.n_heads)]
        )
        self.d_head = d_head
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, sequences):
        # Sequences has shape (N, seq_length, token_dim)
        # We go into shape    (N, seq_length, n_heads, token_dim / n_heads)
        # And come back to    (N, seq_length, item_dim)  (through concatenation)
        result = []
        for sequence in sequences:
            seq_result = []
            for head in range(self.n_heads):
                q_mapping = self.q_mappings[head]
                k_mapping = self.k_mappings[head]
                v_mapping = self.v_mappings[head]

                seq = sequence[:, head * self.d_head : (head + 1) * self.d_head]
                q, k, v = q_mapping(seq), k_mapping(seq), v_mapping(seq)
                scores = q @ k.T / np.sqrt(self.d_head)
                attention = self.softmax(scores)
                seq_result.append(attention @ v)
            result.append(torch.hstack(seq_result))
        return torch.stack(result)

In [ ]:
class MyTransformerBlock(nn.Module):
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(MyTransformerBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        self.norm1 = nn.LayerNorm(hidden_d)
        self.mhsa = MyMSA(hidden_d, n_heads)
        self.norm2 = nn.LayerNorm(hidden_d)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, mlp_ratio * hidden_d),
            nn.GELU(),
            nn.Linear(mlp_ratio * hidden_d, hidden_d),
        )

    def forward(self, x):
        out = x + self.mhsa(self.norm1(x))
        out = out + self.mlp(self.norm2(out))
        return out

In [ ]:
class MyTextTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len,
        hidden_d=64,
        n_heads=4,
        n_blocks=2,
        n_classes=2,
    ):
        super().__init__()

        self.hidden_d = hidden_d
        self.max_len = max_len

        # Token embedding
        self.embedding = nn.Embedding(vocab_size, hidden_d)

        # Positional embedding
        self.register_buffer(
            "pos_embedding",
            get_positional_embeddings(max_len, hidden_d),
            persistent=False,
        )

        # Transformer encoder blocks
        self.blocks = nn.ModuleList(
            [MyTransformerBlock(hidden_d, n_heads) for _ in range(n_blocks)]
        )

        # Classification head
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, n_classes),
            nn.Softmax(dim=-1),
        )

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        emb = self.embedding(x)
        emb = emb + self.pos_embedding.unsqueeze(0)

        # Transformer Blocks
        for block in self.blocks:
            emb = block(emb)
        
        # Getting the classification token only
        cls_token = emb[:, 0]       # first embedding corresponds to <CLS> token - special for classification
        
        return self.mlp(cls_token)

In [ ]:
np.random.seed(0)
torch.manual_seed(0)

# Loading data
texts = [
    "i love transformers",  # positive text
    "this movie is bad",    # negative text
    "pytorch is amazing",   # positive text
    "this is terrible"      # negative text
]
labels = torch.tensor([1, 0, 1, 0])

# Defining model and training options
tokenizer = SimpleTokenizer(texts)
max_len = 6

X = torch.stack([tokenizer.encode(t, max_len) for t in texts])

model = MyTextTransformer(
    vocab_size=len(tokenizer.word2idx),
    max_len=max_len,
    hidden_d=32,
    n_heads=4,
    n_blocks=2,
    n_classes=2,    #number of classes (2 because we have "postive" and "negative" examples)
)

# Training loop
N_EPOCHS = 200
LR = 0.005
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for epoch in trange(N_EPOCHS, desc="Training"):
    optimizer.zero_grad()
    y_hat = model(X)
    loss = criterion(y_hat, labels)
    loss.backward()     # compute gradients (backpropagation) and store them in param.grad (of the optimiser)
    optimizer.step()    # update the net's weights based on the computed gradients

    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Loss {loss.item():.4f}")


Training:   2%|▏         | 4/200 [00:00<00:13, 14.23it/s]

Epoch 0 | Loss 0.6986


Training:  12%|█▎        | 25/200 [00:01<00:06, 25.57it/s]

Epoch 20 | Loss 0.3133


Training:  23%|██▎       | 46/200 [00:01<00:04, 33.14it/s]

...

Training:  84%|████████▎ | 167/200 [00:05<00:00, 34.53it/s]

Epoch 160 | Loss 0.3133


Training:  92%|█████████▏| 183/200 [00:06<00:00, 30.63it/s]

Epoch 180 | Loss 0.3133


Training: 100%|██████████| 200/200 [00:06<00:00, 29.34it/s]


In [30]:

# testing the trained model

test_texts = [
    "i like agentic AI",        # positive text
    "this cookie is not bad"    # positive text
]
X_test = torch.stack([tokenizer.encode(t, max_len) for t in test_texts])
model.eval()
with torch.no_grad():
    sample = X_test
    out = model(sample)
    preds = torch.argmax(out, dim=1)


for text, pred in zip(test_texts, preds):
    print(f"Text: '{text}' -> Predicted class: {pred.item()}")


Text: 'i like agentic AI' -> Predicted class: 1
Text: 'this cookie is not bad' -> Predicted class: 0
